# 02 — ECG Preprocessing Walkthrough
**NANO Study | ESD Lab, University of South Carolina**

> ⚠️ **HIPAA Notice**: Uses synthetic ECG data only. Never commit real ECG files.

This notebook walks through the full ECG cleaning pipeline step-by-step:
1. Simulate a realistic neonatal ECG signal
2. Apply bandpass filter (0.5–40 Hz Butterworth)
3. Detect R-peaks via Pan-Tompkins algorithm
4. Identify and remove artifacts
5. Extract inter-beat intervals (IBI)
6. Validate pipeline against known-good test case

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    import neurokit2 as nk
    NK_AVAILABLE = True
    print(f'NeuroKit2 version: {nk.__version__}')
except ImportError:
    NK_AVAILABLE = False
    print('NeuroKit2 not installed — using scipy fallback for demo.')

print('Setup complete.')

## Step 1: Simulate a Neonatal ECG Signal
We simulate a realistic neonatal ECG at 1024 Hz (HeRO/Actiheart-5 native rate) for ~60 seconds, with a base heart rate of 140 bpm (typical preterm infant) and added artifacts.

In [ ]:
FS = 1024          # Sampling frequency (Hz)
DURATION = 60      # Seconds
HR_BPM = 140       # Baseline HR (typical for VPT infant)
rng = np.random.default_rng(seed=7)

if NK_AVAILABLE:
    ecg_clean = nk.ecg_simulate(duration=DURATION, sampling_rate=FS,
                                 heart_rate=HR_BPM, noise=0.05, random_state=7)
    ecg_noisy = ecg_clean.copy()
else:
    # Minimal fallback: sum of sinusoids approximating ECG-like waveform
    t = np.linspace(0, DURATION, FS * DURATION)
    hr_rad = HR_BPM / 60 * 2 * np.pi
    ecg_clean = (0.8 * np.sin(hr_rad * t) +
                 0.3 * np.sin(3 * hr_rad * t + 0.5) -
                 0.15 * np.sin(5 * hr_rad * t + 1.0))
    ecg_noisy = ecg_clean.copy()

# Add artifacts: burst noise at seconds 20-22 and a baseline drift
artifact_start = int(20 * FS)
artifact_end   = int(22 * FS)
ecg_noisy[artifact_start:artifact_end] += rng.normal(0, 3.0, artifact_end - artifact_start)
# Baseline wander (low-frequency drift)
t_full = np.linspace(0, DURATION, FS * DURATION)
ecg_noisy += 0.4 * np.sin(2 * np.pi * 0.2 * t_full)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(t_full[:FS*5], ecg_clean[:FS*5], lw=0.8, color='royalblue', label='Clean')
axes[0].set_title('Clean Synthetic Neonatal ECG (first 5 s)')
axes[0].set_ylabel('Amplitude (mV)')
axes[0].legend()
axes[1].plot(t_full[:FS*5], ecg_noisy[:FS*5], lw=0.8, color='tomato', label='With artifacts')
axes[1].set_title('ECG with Baseline Wander + Burst Noise')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude (mV)')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Signal length: {len(ecg_noisy)} samples = {len(ecg_noisy)/FS:.1f} s at {FS} Hz')

## Step 2: Bandpass Filter (0.5–40 Hz, 4th-order Butterworth)

In [ ]:
def bandpass_filter(ecg: np.ndarray, fs: int, lowcut: float = 0.5,
                    highcut: float = 40.0, order: int = 4) -> np.ndarray:
    """4th-order Butterworth bandpass filter."""
    nyq = fs / 2
    b, a = scipy_signal.butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return scipy_signal.filtfilt(b, a, ecg)

ecg_filtered = bandpass_filter(ecg_noisy, FS)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_full[:FS*10], ecg_noisy[:FS*10], lw=0.6, alpha=0.6, color='tomato', label='Noisy')
ax.plot(t_full[:FS*10], ecg_filtered[:FS*10], lw=1.0, color='royalblue', label='Filtered')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (mV)')
ax.set_title('Bandpass Filter Effect (0.5–40 Hz) — First 10 s')
ax.legend()
plt.tight_layout()
plt.show()

## Step 3: R-Peak Detection (Pan-Tompkins / NeuroKit2)

In [ ]:
if NK_AVAILABLE:
    _, info = nk.ecg_peaks(ecg_filtered, sampling_rate=FS, method='pantompkins1985')
    r_peaks = info['ECG_R_Peaks']
else:
    # Simple threshold-based peak finding as fallback
    from scipy.signal import find_peaks
    threshold = 0.5 * np.max(ecg_filtered)
    r_peaks, _ = find_peaks(ecg_filtered, height=threshold,
                              distance=int(0.3 * FS))  # min 300ms between beats
    r_peaks = np.array(r_peaks)

ibis_ms = np.diff(r_peaks) / FS * 1000  # IBI in milliseconds
print(f'Detected {len(r_peaks)} R-peaks')
print(f'Mean IBI: {ibis_ms.mean():.1f} ms  (HR = {60000/ibis_ms.mean():.0f} bpm)')
print(f'IBI range: {ibis_ms.min():.1f} – {ibis_ms.max():.1f} ms')

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t_full[:FS*5], ecg_filtered[:FS*5], lw=0.8, color='royalblue', label='Filtered ECG')
peaks_in_window = r_peaks[r_peaks < FS*5]
ax.scatter(peaks_in_window / FS, ecg_filtered[peaks_in_window],
           color='red', zorder=5, s=30, label='R-peaks')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'R-Peak Detection — First 5 s ({len(peaks_in_window)} peaks shown)')
ax.legend()
plt.tight_layout()
plt.show()

## Step 4: Artifact Removal

**Rules** (per `src/preprocessing/ecg_preprocessing.py`):
- Physiologically implausible IBIs: < 300 ms or > 1500 ms → mark artifact
- Statistical outliers: |IBI - local_median| > 3.5 × local_MAD → mark artifact
- Window rejection: > 5 contiguous missing beats OR > 10% missing → exclude window

In [ ]:
def remove_ibi_artifacts(ibis: np.ndarray,
                          min_ibi: float = 300.0, max_ibi: float = 1500.0,
                          z_thresh: float = 3.5, window: int = 31) -> np.ndarray:
    """Mark artifactual IBIs as NaN using physiological + statistical criteria."""
    cleaned = ibis.astype(float).copy()
    # 1. Physiological bounds
    cleaned[(cleaned < min_ibi) | (cleaned > max_ibi)] = np.nan
    # 2. Local median/MAD outlier detection
    for i in range(len(cleaned)):
        lo, hi = max(0, i - window//2), min(len(cleaned), i + window//2 + 1)
        local = cleaned[lo:hi]
        valid = local[~np.isnan(local)]
        if len(valid) < 5:
            continue
        med = np.median(valid)
        mad = np.median(np.abs(valid - med))
        if not np.isnan(cleaned[i]) and mad > 0:
            if np.abs(cleaned[i] - med) > z_thresh * mad * 1.4826:
                cleaned[i] = np.nan
    return cleaned

ibis_clean = remove_ibi_artifacts(ibis_ms)
n_artifacts = np.sum(np.isnan(ibis_clean))
pct_missing = 100 * n_artifacts / len(ibis_clean)
print(f'IBIs: {len(ibis_ms)} total | {n_artifacts} artifacts ({pct_missing:.1f}% missing)')

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
t_ibi = np.cumsum(ibis_ms) / 1000
axes[0].plot(t_ibi, ibis_ms, lw=0.8, color='steelblue', label='Raw IBI')
axes[0].set_ylabel('IBI (ms)')
axes[0].set_title('Raw IBI Series')
axes[0].legend()
artifact_mask = np.isnan(ibis_clean)
axes[1].plot(t_ibi, ibis_clean, lw=0.8, color='forestgreen', label='Cleaned IBI')
axes[1].scatter(t_ibi[artifact_mask], ibis_ms[artifact_mask],
                color='red', s=20, zorder=5, label='Artifacts')
axes[1].set_ylabel('IBI (ms)')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Artifact-Corrected IBI Series')
axes[1].legend()
plt.tight_layout()
plt.show()

## Step 5: Window Rejection Quality Gate

In [ ]:
def check_window_quality(ibis: np.ndarray, max_consecutive: int = 5,
                          max_missing_pct: float = 10.0) -> dict:
    """Evaluate whether a window of IBIs passes quality criteria."""
    n_missing = int(np.sum(np.isnan(ibis)))
    pct_missing = 100.0 * n_missing / len(ibis)
    # Count max consecutive NaN run
    max_consec = 0
    consec = 0
    for v in ibis:
        if np.isnan(v):
            consec += 1
            max_consec = max(max_consec, consec)
        else:
            consec = 0
    passes = (max_consec <= max_consecutive) and (pct_missing <= max_missing_pct)
    return {'passes': passes, 'pct_missing': round(pct_missing, 2),
            'max_consecutive_missing': max_consec}

# Test on our cleaned IBIs
quality = check_window_quality(ibis_clean)
print(f'Window quality check: {quality}')
print(f'Window PASSES quality gate: {quality["passes"]}')

## Summary
The pipeline successfully:
- Applied 4th-order Butterworth bandpass (0.5–40 Hz)
- Detected R-peaks with Pan-Tompkins algorithm
- Removed physiologically implausible IBIs and statistical outliers
- Validated window quality against NANO study thresholds

**Next**: See `03_hrv_feature_engineering.ipynb` for HRV feature extraction and HDA phase identification.